# Django, MTV Architecture & Settings

## Introduction to Django Architecture
---

In this lesson, you'll learn **how Django works internally** and how to configure it.

Understanding the MTV pattern and settings is essential for building robust Django applications.

1. [MTV Architecture](#MTV-Architecture),
    - [What is MTV](#What-is-MTV),
    - [MTV vs MVC](#MTV-vs-MVC),
2. [Request/Response Cycle](#Request/Response-Cycle),
    - [How a Request Flows Through Django](#How-a-Request-Flows-Through-Django),
    - [Step-by-Step Walkthrough](#Step-by-Step-Walkthrough),
3. [Django Settings](#Django-Settings),
    - [Key Settings Explained](#Key-Settings-Explained),
    - [Security Settings](#Security-Settings),
4. [Multiple Settings Files](#Multiple-Settings-Files),
    - [Environment-Specific Configuration](#Environment-Specific-Configuration),
    - [Using Environment Variables](#Using-Environment-Variables),
5. [🧠 EXERCISE 🧠, Configure Your Django Project](#🧠-EXERCISE-🧠,-Configure-Your-Django-Project).

<br>

## MTV Architecture

---

### What is MTV

---

Django follows the **MTV** architectural pattern:

<br>

| Component | Responsibility | Files |
|-----------|---------------|-------|
| **M**odel | Data structure and database interaction | `models.py` |
| **T**emplate | Presentation layer (HTML) | `templates/*.html` |
| **V**iew | Business logic, connects Model and Template | `views.py` |

<br>

Think of it like a restaurant:
- **Model** = Kitchen (prepares the ingredients/data)
- **View** = Waiter (takes orders, brings food, handles requests)
- **Template** = Plate presentation (how the food looks)

<br>

**Model** - Defines your data structure:

In [ ]:
# models.py - Defines data structure

from django.db import models

class Book(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    price = models.DecimalField(max_digits=10, decimal_places=2)
    
    def __str__(self) -> str:
        return self.title

<br>

**View** - Handles the request and prepares data:

In [ ]:
# views.py - Handles business logic

from django.shortcuts import render
from .models import Book

def book_list(request):
    books = Book.objects.all()  # Get data from Model
    context = {'books': books}  # Prepare data for Template
    return render(request, 'catalog/book_list.html', context)

<br>

**Template** - Renders the HTML:

In [ ]:
# templates/catalog/book_list.html (not Python, but HTML with Django template language)

template_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Book List</title>
</head>
<body>
    <h1>Available Books</h1>
    <ul>
        {% for book in books %}
            <li>{{ book.title }} by {{ book.author }} - ${{ book.price }}</li>
        {% endfor %}
    </ul>
</body>
</html>
"""
print(template_content)

<br>

### MTV vs MVC

---

You might have heard of **MVC** (Model-View-Controller), a similar pattern used by other frameworks.

<br>

| MVC | Django MTV | Purpose |
|-----|------------|--------|
| Model | **Model** | Data and database |
| View | **Template** | Presentation (HTML) |
| Controller | **View** | Business logic |

<br>

**The naming is confusing!** In Django:
- The "View" in MVC is called "Template" in Django
- The "Controller" in MVC is called "View" in Django

<br>

Don't worry too much about the naming. Just remember:
- **Models** = Data
- **Views** = Logic (the "waiter")
- **Templates** = HTML presentation

<br>

## Request/Response Cycle

---

### How a Request Flows Through Django

---

When a user visits your website, here's what happens:

```
User Browser                                          Django Application
     │                                                        │
     │  1. HTTP Request (GET /books/)                        │
     │ ─────────────────────────────────────────────────────► │
     │                                                        │
     │                        2. URL Router (urls.py)         │
     │                        Matches URL to View             │
     │                               │                        │
     │                               ▼                        │
     │                        3. View (views.py)              │
     │                        Processes request               │
     │                               │                        │
     │                        4. Model (models.py)            │
     │                        Fetches data from DB            │
     │                               │                        │
     │                        5. Template (*.html)            │
     │                        Renders HTML                    │
     │                               │                        │
     │  6. HTTP Response (HTML page)                         │
     │ ◄───────────────────────────────────────────────────── │
     │                                                        │
```

<br>

### Step-by-Step Walkthrough

---

Let's trace a request for `http://localhost:8000/books/` step by step:

<br>

**Step 1: User makes request**

Browser sends HTTP request to Django server.

<br>

**Step 2: URL routing** (`urls.py`)

In [ ]:
# urls.py - Maps URL to view function

from django.urls import path
from catalog import views

urlpatterns = [
    path('books/', views.book_list, name='book_list'),
    #    ^^^^^^^^  ^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^
    #    URL path   View function    URL name
]

<br>

**Step 3: View processes request** (`views.py`)

In [ ]:
# views.py - Handles business logic

from django.shortcuts import render
from .models import Book

def book_list(request):
    # request object contains all HTTP request data
    print(f"Method: {request.method}")  # GET, POST, etc.
    print(f"Path: {request.path}")      # /books/
    print(f"User: {request.user}")      # Current user
    
    # Step 4: Get data from Model
    books = Book.objects.all()
    
    # Step 5: Render Template with data
    return render(request, 'catalog/book_list.html', {'books': books})

<br>

**Step 4: Model fetches data** (`models.py`)

In [ ]:
# models.py - Django ORM queries the database

# Book.objects.all() generates SQL:
# SELECT * FROM catalog_book;

# Book.objects.filter(price__lt=20) generates:
# SELECT * FROM catalog_book WHERE price < 20;

<br>

**Step 5: Template renders HTML**

In [ ]:
# Template receives 'books' from view and generates HTML
# {% for book in books %} loops through all books
# {{ book.title }} outputs the book's title

# The resulting HTML is sent back to the browser

<br>

**Step 6: Response sent to browser**

Django sends the rendered HTML as an HTTP response. Browser displays the page.

<br>

## Django Settings

---

### Key Settings Explained

---

The `settings.py` file controls how Django behaves. Here are the most important settings:

<br>

**DEBUG**

In [ ]:
# DEBUG mode - detailed error pages, auto-reload

DEBUG = True   # Development: shows detailed error pages
DEBUG = False  # Production: shows generic error pages

**IMPORTANT:** Never run with `DEBUG = True` in production! It exposes sensitive information.

<br>

**ALLOWED_HOSTS**

In [ ]:
# Which domains can serve this site

ALLOWED_HOSTS = []  # Development (only works with DEBUG=True)

ALLOWED_HOSTS = [
    'example.com',
    'www.example.com',
    'localhost',
    '127.0.0.1',
]  # Production

<br>

**DATABASES**

In [ ]:
# Database configuration

# Default: SQLite (file-based, great for development)
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',
        'NAME': BASE_DIR / 'db.sqlite3',
    }
}

# PostgreSQL (recommended for production)
DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': 'mydatabase',
        'USER': 'myuser',
        'PASSWORD': 'mypassword',
        'HOST': 'localhost',
        'PORT': '5432',
    }
}

<br>

**STATIC_URL and STATICFILES_DIRS**

In [ ]:
# Static files (CSS, JavaScript, images)

STATIC_URL = '/static/'  # URL prefix for static files

# Additional directories to look for static files
STATICFILES_DIRS = [
    BASE_DIR / 'static',
]

# Where to collect static files for production
STATIC_ROOT = BASE_DIR / 'staticfiles'

<br>

**TEMPLATES**

In [ ]:
# Template configuration

TEMPLATES = [
    {
        'BACKEND': 'django.template.backends.django.DjangoTemplates',
        'DIRS': [BASE_DIR / 'templates'],  # Project-wide templates
        'APP_DIRS': True,  # Look in app/templates/ directories
        'OPTIONS': {
            'context_processors': [
                'django.template.context_processors.debug',
                'django.template.context_processors.request',
                'django.contrib.auth.context_processors.auth',
                'django.contrib.messages.context_processors.messages',
            ],
        },
    },
]

<br>

### Security Settings

---

Django provides several security settings that you should configure for production:

<br>

In [ ]:
# SECRET_KEY - Used for cryptographic signing
# NEVER commit this to version control!
SECRET_KEY = 'django-insecure-your-key-here'  # BAD: Hardcoded

import os
SECRET_KEY = os.environ.get('DJANGO_SECRET_KEY')  # GOOD: From environment

In [ ]:
# Security settings for production

# HTTPS settings
SECURE_SSL_REDIRECT = True          # Redirect HTTP to HTTPS
SESSION_COOKIE_SECURE = True        # Only send session cookie over HTTPS
CSRF_COOKIE_SECURE = True           # Only send CSRF cookie over HTTPS

# HSTS (HTTP Strict Transport Security)
SECURE_HSTS_SECONDS = 31536000      # 1 year
SECURE_HSTS_INCLUDE_SUBDOMAINS = True
SECURE_HSTS_PRELOAD = True

# Other security settings
SECURE_CONTENT_TYPE_NOSNIFF = True  # Prevent MIME type sniffing
X_FRAME_OPTIONS = 'DENY'            # Prevent clickjacking

<br>

**Quick reference table:**

| Setting | Development | Production |
|---------|-------------|------------|
| `DEBUG` | `True` | `False` |
| `SECRET_KEY` | Can be hardcoded | Environment variable |
| `ALLOWED_HOSTS` | `[]` | Your domains |
| `SECURE_SSL_REDIRECT` | `False` | `True` |
| Database | SQLite | PostgreSQL |

<br>

## Multiple Settings Files

---

### Environment-Specific Configuration

---

A common pattern is to split settings into multiple files:

```
mysite/
├── settings/
│   ├── __init__.py
│   ├── base.py        # Shared settings
│   ├── development.py # Development-specific
│   └── production.py  # Production-specific
```

<br>

**base.py** - Common settings:

In [ ]:
# settings/base.py - Shared settings

from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent.parent

INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    # ... other apps
]

# Settings that are the same across environments
LANGUAGE_CODE = 'en-us'
TIME_ZONE = 'UTC'
USE_I18N = True
USE_TZ = True

<br>

**development.py** - Development-specific:

In [ ]:
# settings/development.py

from .base import *  # Import all base settings

DEBUG = True

SECRET_KEY = 'django-insecure-dev-key-not-for-production'

ALLOWED_HOSTS = ['localhost', '127.0.0.1']

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.sqlite3',
        'NAME': BASE_DIR / 'db.sqlite3',
    }
}

# Debug toolbar for development
INSTALLED_APPS += ['debug_toolbar']

<br>

**production.py** - Production-specific:

In [ ]:
# settings/production.py

import os
from .base import *  # Import all base settings

DEBUG = False

SECRET_KEY = os.environ.get('DJANGO_SECRET_KEY')

ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', '').split(',')

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': os.environ.get('DB_NAME'),
        'USER': os.environ.get('DB_USER'),
        'PASSWORD': os.environ.get('DB_PASSWORD'),
        'HOST': os.environ.get('DB_HOST'),
        'PORT': os.environ.get('DB_PORT', '5432'),
    }
}

# Security settings
SECURE_SSL_REDIRECT = True
SESSION_COOKIE_SECURE = True
CSRF_COOKIE_SECURE = True

<br>

**How to use:**

```bash
# Development
export DJANGO_SETTINGS_MODULE=mysite.settings.development
python manage.py runserver

# Production
export DJANGO_SETTINGS_MODULE=mysite.settings.production
gunicorn mysite.wsgi
```

<br>

### Using Environment Variables

---

A simpler approach is using environment variables with a single settings file.

The `python-decouple` package makes this easy:

```bash
pip install python-decouple
```

In [ ]:
# settings.py with python-decouple

from decouple import config, Csv

SECRET_KEY = config('SECRET_KEY')
DEBUG = config('DEBUG', default=False, cast=bool)
ALLOWED_HOSTS = config('ALLOWED_HOSTS', default='', cast=Csv())

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': config('DB_NAME'),
        'USER': config('DB_USER'),
        'PASSWORD': config('DB_PASSWORD'),
        'HOST': config('DB_HOST', default='localhost'),
        'PORT': config('DB_PORT', default='5432'),
    }
}

<br>

Create a `.env` file (add to `.gitignore`!):

```ini
# .env file
SECRET_KEY=your-super-secret-key-here
DEBUG=True
ALLOWED_HOSTS=localhost,127.0.0.1
DB_NAME=mydb
DB_USER=myuser
DB_PASSWORD=mypassword
DB_HOST=localhost
DB_PORT=5432
```

<br>

**Benefits of this approach:**

| Benefit | Description |
|---------|-------------|
| **Security** | Secrets not in code |
| **Flexibility** | Same code, different config |
| **12-Factor App** | Follows best practices |
| **Simple** | One settings file |

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **MTV** | Model (data), Template (HTML), View (logic) |
| **Request flow** | URL → View → Model → Template → Response |
| **DEBUG** | `True` for development, `False` for production |
| **SECRET_KEY** | Keep it secret, use environment variables |
| **ALLOWED_HOSTS** | List of valid domain names |
| **DATABASES** | SQLite for dev, PostgreSQL for production |
| **Multiple settings** | Split by environment or use .env files |

<br>

### 🧠 EXERCISE 🧠, Configure Your Django Project

---

Using the `bookstore` project:

1. Install `python-decouple`: `pip install python-decouple`
2. Create a `.env` file in the project root with:
   - `SECRET_KEY` (generate a new one or use the existing)
   - `DEBUG=True`
3. Update `settings.py` to use `config()` for `SECRET_KEY` and `DEBUG`
4. Add `.env` to `.gitignore`
5. Verify the project still works: `python manage.py check`

<details>
    <summary>▶️ Solution</summary>
    
**1. Install python-decouple:**

```bash
pip install python-decouple
pip freeze > requirements.txt  # Update requirements
```

**2. Create `.env` file:**

```ini
# .env
SECRET_KEY=django-insecure-your-existing-key-from-settings
DEBUG=True
```

**3. Update `bookstore_project/settings.py`:**

```python
# At the top, add:
from decouple import config

# Replace these lines:
SECRET_KEY = config('SECRET_KEY')
DEBUG = config('DEBUG', default=False, cast=bool)
```

**4. Create/update `.gitignore`:**

```gitignore
# .gitignore
.env
.venv/
*.pyc
__pycache__/
db.sqlite3
```

**5. Verify:**

```bash
python manage.py check
# Expected: System check identified no issues (0 silenced).
```
</details>

<br>

### 🧠 EXERCISE 🧠, Trace the Request Flow

---

Create a simple view to understand the MTV flow:

1. In `catalog/views.py`, create a `home` view that returns an HttpResponse with "Welcome to Bookstore!"
2. In `catalog/urls.py`, add a path for the home view at the root (`''`)
3. In `bookstore_project/urls.py`, include the catalog URLs at the root
4. Run the server and visit `http://127.0.0.1:8000/`
5. Verify you see "Welcome to Bookstore!"

<details>
    <summary>▶️ Solution</summary>
    
**1. Edit `catalog/views.py`:**

```python
from django.http import HttpResponse

def home(request):
    return HttpResponse("Welcome to Bookstore!")
```

**2. Edit `catalog/urls.py`:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('', views.home, name='home'),
]
```

**3. Edit `bookstore_project/urls.py`:**

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('catalog.urls')),  # Include at root
]
```

**4-5. Run and test:**

```bash
python manage.py runserver
# Visit http://127.0.0.1:8000/
# You should see: "Welcome to Bookstore!"
```

**What happened:**
1. Browser requested `/`
2. `bookstore_project/urls.py` matched `''` → included `catalog.urls`
3. `catalog/urls.py` matched `''` → called `views.home`
4. `views.home` returned `HttpResponse("Welcome to Bookstore!")`
5. Browser displayed the response
</details>

---